# Compliance Reporting for ML Systems

## What You Will Learn
- Understand GDPR and regulatory compliance requirements for ML
- Implement Right to Explanation and Data Subject Access Requests (DSAR)
- Generate Model Cards from MLflow metadata
- Create compliance report templates

## Why This Matters in MLOps
Regulations like GDPR impose strict requirements on automated decision-making. Organizations must be able to explain model decisions, delete user data on request, and document model provenance. Compliance is not optional — it is a legal requirement with significant penalties for non-compliance.

## You're Done When...
- You can explain key GDPR requirements for ML systems
- You have generated a Model Card from MLflow run metadata
- You have created a compliance report template
- You have simulated a data deletion request
- You have completed the fill-in-the-blank exercises

---

### MLOps Perspective

Compliance in MLOps means building systems that can:
1. **Explain** predictions (Right to Explanation, GDPR Art. 22)
2. **Delete** user data on request (Right to Erasure, GDPR Art. 17)
3. **Document** model purpose, performance, and limitations (Model Cards)
4. **Report** compliance status on demand to auditors

These requirements must be designed into the ML pipeline, not retrofitted.

---
## 1. GDPR and Regulatory Compliance for ML

### Key GDPR Requirements
| Requirement | Article | Implication for ML |
|------------|---------|-------------------|
| Right to Explanation | Art. 22 | Must explain automated decisions |
| Right to Erasure | Art. 17 | Must delete user data from models |
| Data Minimization | Art. 5 | Collect only necessary data |
| Purpose Limitation | Art. 5 | Use data only for stated purpose |
| Accountability | Art. 24 | Document compliance activities |

### Other Regulations
- **HIPAA**: Health data privacy, requires audit controls
- **SOX**: Financial reporting, requires model validation
- **CCPA**: California privacy rights, similar to GDPR

---
## 2. Right to Explanation

GDPR Article 22 gives individuals the right not to be subject to solely automated decisions without explanation. In practice, this means you need:
- Feature importance scores for each prediction
- Model documentation (purpose, training data, accuracy)
- Human review process for high-stakes decisions

In [ ]:
# Simulate a prediction explanation
import json
from datetime import datetime

def generate_explanation(features, prediction, feature_importances):
    """Generate a human-readable explanation for a prediction."""
    top_features = sorted(
        feature_importances.items(),
        key=lambda x: abs(x[1]),
        reverse=True
    )[:3]
    
    explanation = {
        'prediction': prediction,
        'top_contributing_features': [
            {'feature': f, 'importance': round(v, 4)}
            for f, v in top_features
        ],
        'model_type': 'RandomForest',
        'confidence': 0.87,
        'generated_at': datetime.utcnow().isoformat() + 'Z',
        'human_review_available': True
    }
    return explanation

# Example
features = {'temperature': 22, 'humidity': 0.65, 'windspeed': 15, 'season': 'summer'}
importances = {'temperature': 0.42, 'humidity': 0.31, 'windspeed': 0.18, 'season': 0.09}

exp = generate_explanation(features, 4520, importances)
print(json.dumps(exp, indent=2))

---
## 3. Data Subject Access Request (DSAR) Simulation

GDPR Article 15 gives individuals the right to access their data. We simulate finding and deleting user data.

In [ ]:
import pandas as pd
import numpy as np

# Simulate a training dataset with user identifiers
np.random.seed(42)
n_users = 100
user_data = pd.DataFrame({
    'user_id': [f'user_{i:04d}' for i in range(n_users)],
    'age': np.random.randint(18, 70, n_users),
    'city': np.random.choice(['NYC', 'SFO', 'CHI', 'ATX'], n_users),
    'rides_per_week': np.random.randint(0, 30, n_users),
    'is_active': np.random.choice([True, False], n_users, p=[0.8, 0.2])
})

print(f'Dataset: {len(user_data)} users')
print(user_data.head())

In [ ]:
# Simulate Right to Erasure (Article 17)
def handle_deletion_request(user_id, data):
    """Remove user data in response to a deletion request."""
    if user_id not in data['user_id'].values:
        return {'status': 'not_found', 'user_id': user_id}
    
    # Record what was deleted for audit purposes
    user_row = data[data['user_id'] == user_id].iloc[0]
    deletion_record = {
        'status': 'deleted',
        'user_id': user_id,
        'deleted_at': datetime.utcnow().isoformat() + 'Z',
        'deleted_fields': list(data.columns),
        'request_id': f'del_{int(datetime.utcnow().timestamp())}'
    }
    
    # Remove data (in practice, also retrain or mark for exclusion)
    data.drop(data[data['user_id'] == user_id].index, inplace=True)
    return deletion_record

# Process a deletion request
result = handle_deletion_request('user_0042', user_data)
print(json.dumps(result, indent=2))
print(f'\nRemaining users: {len(user_data)}')

---
## 4. Model Cards and Documentation

Model Cards are standardized documentation that communicate model details, intended use, performance, and limitations.

In [ ]:
# Generate a Model Card from MLflow metadata

def generate_model_card(model_name, model_version, run_metrics, params, tags):
    """Generate a Model Card summary from MLflow metadata."""
    card = {
        'model_details': {
            'name': model_name,
            'version': model_version,
            'type': params.get('model_type', 'Unknown'),
            'authors': tags.get('author', 'ML Team'),
            'date': datetime.utcnow().strftime('%Y-%m-%d')
        },
        'intended_use': {
            'primary_use': 'Bike demand forecasting for city planning',
            'out_of_scope': 'Individual trip prediction or real-time navigation',
            'limitations': [
                'May not generalize to cities not in training data',
                'Does not account for special events or holidays',
                'Accuracy decreases during extreme weather'
            ]
        },
        'performance': {
            'metrics': run_metrics,
            'evaluation_data': 'Holdout test set (20% of historical data)',
            'slice_performance': {
                'weekday_rmse': 320,
                'weekend_rmse': 285,
                'holiday_rmse': 410
            }
        },
        'fairness': {
            'tested_dimensions': ['time_of_day', 'season', 'neighborhood_type'],
            'bias_mitigation': 'Stratified sampling during train/test split'
        },
        'governance': {
            'approval_status': tags.get('review_status', 'pending'),
            'reviewed_by': tags.get('reviewed_by', 'N/A'),
            'training_data': 'Bike sharing dataset (2011-2012)',
            'retraining_schedule': 'Quarterly'
        }
    }
    return card

# Example
model_card = generate_model_card(
    model_name='BikeDemandForecaster',
    model_version='v2',
    run_metrics={'accuracy': 0.94, 'rmse': 342.5, 'mae': 210.3},
    params={'model_type': 'RandomForest', 'n_estimators': 200},
    tags={'author': 'data-science-team', 'review_status': 'approved', 'reviewed_by': 'ml-eng@co.com'}
)

print(json.dumps(model_card, indent=2))

---
## 5. Compliance Report Template

Generate a structured compliance report for auditors.

In [ ]:
def generate_compliance_report(model_card, audit_events, deletion_logs):
    """Generate a compliance report for auditors."""
    report = {
        'report_title': 'ML System Compliance Report',
        'generated_at': datetime.utcnow().isoformat() + 'Z',
        'system_name': 'Bike Demand Forecasting System',
        'compliance_status': {
            'gdpr_article_17_right_to_erasure': {
                'status': 'implemented',
                'deletion_requests_processed': len(deletion_logs),
                'evidence': deletion_logs
            },
            'gdpr_article_22_automated_decisions': {
                'status': 'implemented',
                'explanation_method': 'Feature importance (SHAP)',
                'human_review_path': True
            },
            'model_documentation': {
                'status': 'implemented',
                'model_card_available': True,
                'last_updated': model_card['model_details']['date']
            },
            'audit_trail': {
                'status': 'implemented',
                'total_audit_events': len(audit_events),
                'date_range': 'See attached audit log'
            }
        },
        'model_card': model_card,
        'recommendations': [
            'Implement automated bias monitoring in production',
            'Add data retention policies to pipeline documentation',
            'Conduct quarterly fairness audits'
        ]
    }
    return report

# Generate sample compliance report
report = generate_compliance_report(
    model_card,
    audit_events=[{'event_type': 'prediction', 'count': 1500}],
    deletion_logs=[{'user_id': 'user_0042', 'status': 'deleted'}]
)

print(json.dumps(report, indent=2))

---
## 6. Fill-in-the-Blank Exercises

In [ ]:
# Exercise 1: Generate a Model Card

def create_model_card(model_name, metrics):
    card = {
        'model_name': model_name,
        'metrics': ____,  # Hint: the metrics parameter
        'generated_at': datetime.utcnow().isoformat() + 'Z',
        'status': 'documented'
    }
    return card

card = create_model_card('BikeModel', {'accuracy': 0.95, 'rmse': 300})
print(json.dumps(card, indent=2))

In [ ]:
# Exercise 2: Handle a data deletion request (DSAR)

def delete_user_data(dataset, user_id_to_delete):
    """Delete user data and return confirmation."""
    initial_count = len(dataset)
    dataset.____(  # Hint: use drop method
        dataset[dataset['user_id'] == user_id_to_delete].index,
        inplace=True
    )
    final_count = len(dataset)
    return {
        'user_id': user_id_to_delete,
        'records_deleted': initial_count - final_count,
        'status': 'completed'
    }

# Test
test_df = pd.DataFrame({'user_id': ['user_0001', 'user_0002', 'user_0003']})
result = delete_user_data(test_df, 'user_0002')
print(json.dumps(result, indent=2))

---
## Summary

- GDPR requires Right to Explanation (Art. 22), Right to Erasure (Art. 17), and accountability
- Model Cards document model purpose, performance, and limitations
- Compliance reports consolidate evidence for auditors
- Data deletion simulation demonstrates DSAR handling capability

---
## Workshop Complete!

You have completed the ML Security & Compliance advanced lab. You now understand:
1. Security basics: encryption, hashing, secure config
2. Governance: RBAC, model versioning, approval workflows
3. Audit logging: tracking, querying, immutable logs
4. Compliance: GDPR, Model Cards, DSAR, reporting